# BenchmarkPython PAG-Vul binary training

This notebook is deployed by `kaggle_deploy/deploy_kaggle.py`. Its input Dataset contains the graph artifact and the exact trainer source used for this run.

In [ ]:
!pip install -q torch-geometric pennylane


In [ ]:
import json
import subprocess
import torch
from pathlib import Path

WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
DATASET_HANDLE = 'khangtrn2/benchmarkpython-pagvul-binary-assets'

def find_input_dir() -> Path:
    matches = [path for path in INPUT_ROOT.iterdir() if (path / 'run_config.json').is_file()]
    if not matches:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        matches = [downloaded] if (downloaded / 'run_config.json').is_file() else []
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one Kaggle input directory with run_config.json, found {matches}')
    return matches[0]

def resolve_device() -> str:
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable. This notebook requires a compatible Kaggle GPU.')
    try:
        # A visible GPU can still be incompatible with the PyTorch wheel.
        torch.zeros(1, device='cuda').add_(1).item()
    except Exception as exc:
        raise RuntimeError(f'CUDA is unusable ({exc}). Request a T4 accelerator and rerun.') from exc
    return 'cuda'

INPUT = find_input_dir()
config = json.loads((INPUT / 'run_config.json').read_text())
attention = config['attention']
device = resolve_device()
print(f'Running {attention} PAG-Vul binary training from {INPUT.name} on {device}')

command = [
    'python', str(INPUT / 'train_pagvul_binary.py'),
    '--dataset', str(INPUT / 'benchmarkpython_binary_graphs.pt'),
    '--attention', attention,
    '--device', device,
    '--out-dir', str(WORK / f'pagvul_binary_{attention}'),
]
for name, value in config.get('trainer_args', {}).items():
    command.extend(['--' + name.replace('_', '-'), str(value)])
subprocess.run(command, check=True)


In [ ]:
report = WORK / f'pagvul_binary_{attention}' / 'report.json'
print(report.read_text())
